# Polyglot-120: train, evaluate, and publish

**Before running:** Runtime → Change runtime type → **T4 GPU**. Set `HF_USERNAME` below. Then Runtime → **Run all**.

The login prompt wants a Hugging Face *write* token (hf.co/settings/tokens). It is only needed for the final upload step.

Total time on a T4: about 45 minutes. The evaluation table at the end gives the real numbers for your README and email.

If a step fails, the notebook stops at that cell on purpose. Scroll up inside that cell for the traceback.

In [ ]:
HF_USERNAME = "your-hf-username"   # <-- change this
NUM_LANGS = 120
EPOCHS = 1


In [ ]:
!nvidia-smi -L || echo 'No GPU: switch runtime to T4 before continuing'
!pip -q install 'transformers>=4.44' 'datasets>=2.20' 'accelerate>=0.33' scikit-learn pandas sentencepiece protobuf huggingface_hub
import transformers, torch; print('transformers', transformers.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
import os
for d in ('scripts','data','reports'): os.makedirs(d, exist_ok=True)


In [ ]:
%%writefile scripts/prepare_data.py
"""
Prepare training and evaluation data for Polyglot-120 from FLORES-200.

FLORES-200 gives ~1,000 parallel sentences per language for 200+ languages
in two splits: dev (used here for training) and devtest (used for testing).

Data source, in order:
  1. Meta's public tarball (no account, no gating):
       https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz
  2. Fallback: the Hugging Face dataset facebook/flores (gated; needs login
     and clicking "Agree" on the dataset page).

Outputs (in --out):
  train.csv              text,label           (dev split)
  test_clean.csv         text,label,slice     (devtest, unmodified)
  test_short.csv         truncated inputs (2-5 words, or 6-12 chars for unspaced scripts)
  test_noisy.csv         character-level noise: drops, swaps, duplicates, case flips
  test_code_mixed.csv    70% of a sentence in label language + 30% from another language
  test_mixed_script.csv  like code_mixed, but the other language uses a different script
  labels.json            sorted list of language codes

Language selection: every MUST_HAVE language that exists in FLORES-200 (about 70
widely used languages, English included) plus round-robin picks across scripts up
to --num_langs, so the set is both useful and script-diverse.

Usage:
  python scripts/prepare_data.py --out data --num_langs 120
  python scripts/prepare_data.py --out data --num_langs 0      # all languages
"""
import argparse
import csv
import io
import json
import os
import random
import tarfile
import urllib.request
from collections import defaultdict

TARBALL_URLS = [
    "https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz",
    "https://tinyurl.com/flores200dataset",
]

# Always included (when present in FLORES-200): the most widely used languages, so the
# label set is useful as a product, not just as a benchmark. The remaining slots are
# filled round-robin across scripts for diversity (see pick_languages).
MUST_HAVE = [
    "eng_Latn", "zho_Hans", "zho_Hant", "hin_Deva", "spa_Latn", "fra_Latn", "arb_Arab",
    "ben_Beng", "por_Latn", "rus_Cyrl", "urd_Arab", "ind_Latn", "deu_Latn", "jpn_Jpan",
    "swh_Latn", "mar_Deva", "tel_Telu", "tur_Latn", "tam_Taml", "vie_Latn", "kor_Hang",
    "ita_Latn", "pes_Arab", "pol_Latn", "ukr_Cyrl", "nld_Latn", "tha_Thai", "guj_Gujr",
    "kan_Knda", "mal_Mlym", "mya_Mymr", "pan_Guru", "ron_Latn", "hun_Latn", "ces_Latn",
    "ell_Grek", "swe_Latn", "heb_Hebr", "fin_Latn", "dan_Latn", "nob_Latn", "zsm_Latn",
    "tgl_Latn", "hau_Latn", "yor_Latn", "ibo_Latn", "amh_Ethi", "som_Latn", "khm_Khmr",
    "lao_Laoo", "sin_Sinh", "npi_Deva", "kat_Geor", "hye_Armn", "azj_Latn", "kaz_Cyrl",
    "uzn_Latn", "bul_Cyrl", "srp_Cyrl", "hrv_Latn", "slk_Latn", "slv_Latn", "lit_Latn",
    "lvs_Latn", "est_Latn", "cat_Latn", "afr_Latn", "jav_Latn", "ceb_Latn", "zul_Latn",
]


# ----------------------------------------------------------------------------
# Loading
# ----------------------------------------------------------------------------
def load_from_tarball(cache_dir):
    """Download Meta's public FLORES-200 tarball and parse dev/devtest files.
    Returns {"dev": {lang: [sentences]}, "devtest": {lang: [sentences]}}."""
    os.makedirs(cache_dir, exist_ok=True)
    local = os.path.join(cache_dir, "flores200_dataset.tar.gz")
    if not os.path.exists(local):
        last_err = None
        for url in TARBALL_URLS:
            try:
                print(f"downloading {url} ...")
                urllib.request.urlretrieve(url, local)
                break
            except Exception as e:  # try next mirror
                last_err = e
                print("  failed:", e)
        if not os.path.exists(local):
            raise RuntimeError(f"could not download FLORES-200 tarball: {last_err}")
    splits = {"dev": defaultdict(list), "devtest": defaultdict(list)}
    with tarfile.open(local, "r:gz") as tf:
        for m in tf.getmembers():
            if not m.isfile():
                continue
            name = m.name.replace("\\", "/")
            parts = name.split("/")
            if len(parts) < 2:
                continue
            split, fname = parts[-2], parts[-1]
            if split not in splits:
                continue
            if not (fname.endswith(".dev") or fname.endswith(".devtest")):
                continue
            lang = fname.rsplit(".", 1)[0]  # e.g. eng_Latn
            text = io.TextIOWrapper(tf.extractfile(m), encoding="utf-8").read()
            splits[split][lang] = [ln.strip() for ln in text.splitlines() if ln.strip()]
    if not splits["dev"] or not splits["devtest"]:
        raise RuntimeError("tarball parsed but no dev/devtest files found")
    print(f"tarball: {len(splits['dev'])} languages in dev, {len(splits['devtest'])} in devtest")
    return splits


def load_from_hub():
    """Fallback: gated Hub dataset. Requires `huggingface-cli login` and
    accepting the terms at https://huggingface.co/datasets/facebook/flores"""
    from datasets import load_dataset
    ds = load_dataset("facebook/flores", "all")
    splits = {"dev": defaultdict(list), "devtest": defaultdict(list)}
    for split in splits:
        cols = [c for c in ds[split].column_names if c.startswith("sentence_")]
        for row in ds[split]:
            for c in cols:
                t = row[c]
                if t and t.strip():
                    splits[split][c[len("sentence_"):]].append(t.strip())
    return splits


def load_flores(cache_dir):
    try:
        return load_from_tarball(cache_dir)
    except Exception as e:
        print("tarball route failed, trying Hugging Face Hub:", e)
        return load_from_hub()


# ----------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------
def pick_languages(all_langs, n, rng, must_have=()):
    """Pick n languages: every available MUST_HAVE language first, then fill the
    remaining slots round-robin across scripts so the set isn't all Latin-script."""
    if n <= 0 or n >= len(all_langs):
        return sorted(all_langs)
    avail = set(all_langs)
    chosen = [l for l in must_have if l in avail][:n]
    missing = [l for l in must_have if l not in avail]
    if missing:
        print(f"note: {len(missing)} must-have languages not in this FLORES release: {missing}")
    by_script = defaultdict(list)
    for code in all_langs:
        if code not in chosen:
            by_script[code.split("_")[-1]].append(code)
    for v in by_script.values():
        rng.shuffle(v)
    scripts = sorted(by_script.keys())
    while len(chosen) < n:
        progressed = False
        for s in scripts:
            if by_script[s]:
                chosen.append(by_script[s].pop())
                progressed = True
                if len(chosen) == n:
                    break
        if not progressed:
            break
    return sorted(chosen)


def has_spaces(text):
    return " " in text.strip()


def make_short(text, rng):
    if has_spaces(text):
        words = text.split()
        return " ".join(words[: rng.randint(2, min(5, len(words)))])
    return text[: rng.randint(6, min(12, len(text)))]


def make_noisy(text, rng, p=0.08):
    chars, out, i = list(text), [], 0
    while i < len(chars):
        c, r = chars[i], rng.random()
        if r < p / 3:
            i += 1  # drop
            continue
        if r < 2 * p / 3 and i + 1 < len(chars):
            out.extend([chars[i + 1], c])  # swap
            i += 2
            continue
        if r < p:
            out.extend([c, c])  # duplicate
            i += 1
            continue
        if c.isalpha() and rng.random() < 0.05:
            c = c.upper() if c.islower() else c.lower()
        out.append(c)
        i += 1
    return "".join(out)


def mix(text_a, text_b, rng, frac_a=0.7):
    """~frac_a of A's tokens followed by ~(1-frac_a) of B's tokens."""
    a = text_a.split() if has_spaces(text_a) else list(text_a)
    b = text_b.split() if has_spaces(text_b) else list(text_b)
    ka, kb = max(1, int(len(a) * frac_a)), max(1, int(len(b) * (1 - frac_a)))
    ja, jb = (" " if has_spaces(text_a) else ""), (" " if has_spaces(text_b) else "")
    return ja.join(a[:ka]) + " " + jb.join(b[:kb])


def write_csv(path, rows, header):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)


# ----------------------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="data")
    ap.add_argument("--num_langs", type=int, default=120, help="0 = all languages")
    ap.add_argument("--seed", type=int, default=13)
    ap.add_argument("--per_lang_test", type=int, default=300,
                    help="devtest sentences per language used for each robustness slice")
    ap.add_argument("--cache", default=".flores_cache")
    ap.add_argument("--must_have", type=int, default=1,
                    help="1 = always include the MUST_HAVE major languages first (default), 0 = pure round-robin")
    args = ap.parse_args()

    rng = random.Random(args.seed)
    os.makedirs(args.out, exist_ok=True)

    splits = load_flores(args.cache)
    all_langs = sorted(set(splits["dev"]) & set(splits["devtest"]))
    langs = pick_languages(all_langs, args.num_langs, rng, MUST_HAVE if args.must_have else ())
    print(f"Using {len(langs)} of {len(all_langs)} available languages")

    # ---- train: dev split ----
    train_rows = [(t, lang) for lang in langs for t in splits["dev"][lang]]
    rng.shuffle(train_rows)
    write_csv(os.path.join(args.out, "train.csv"), train_rows, ["text", "label"])
    print(f"train: {len(train_rows)} rows")

    # ---- test: devtest split + robustness slices ----
    per_lang = {lang: list(splits["devtest"][lang]) for lang in langs}
    script_of = {l: l.split("_")[-1] for l in langs}
    clean, short, noisy, code_mixed, mixed_script = [], [], [], [], []
    for lang in langs:
        sents = per_lang[lang]
        rng.shuffle(sents)
        clean.extend((t, lang, "clean") for t in sents)
        for t in sents[: args.per_lang_test]:
            short.append((make_short(t, rng), lang, "short"))
            noisy.append((make_noisy(t, rng), lang, "noisy"))
            other = rng.choice([l for l in langs if l != lang])
            code_mixed.append((mix(t, rng.choice(per_lang[other]), rng), lang, "code_mixed"))
            diff = [l for l in langs if script_of[l] != script_of[lang]]
            if diff:
                o2 = rng.choice(diff)
                mixed_script.append((mix(t, rng.choice(per_lang[o2]), rng), lang, "mixed_script"))

    hdr = ["text", "label", "slice"]
    for name, rows in [("clean", clean), ("short", short), ("noisy", noisy),
                       ("code_mixed", code_mixed), ("mixed_script", mixed_script)]:
        write_csv(os.path.join(args.out, f"test_{name}.csv"), rows, hdr)
    with open(os.path.join(args.out, "labels.json"), "w") as f:
        json.dump(langs, f, indent=2)
    print("test slices:", len(clean), len(short), len(noisy), len(code_mixed), len(mixed_script))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/train.py
"""
Fine-tune a multilingual transformer for language identification.

Default backbone: xlm-roberta-base (278M params, 100 pretraining languages).
On a free Colab T4 with ~120 languages x ~1,000 sentences, one epoch at
max_length=64 takes roughly 15-30 minutes.

Usage:
  python scripts/train.py --data data --out model --epochs 1
"""
import argparse
import inspect
import json
import math
import os

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import f1_score
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--out", default="model")
    ap.add_argument("--model_name", default="xlm-roberta-base")
    ap.add_argument("--epochs", type=float, default=1.0)
    ap.add_argument("--batch_size", type=int, default=64)
    ap.add_argument("--lr", type=float, default=3e-5)
    ap.add_argument("--max_length", type=int, default=64)
    ap.add_argument("--val_frac", type=float, default=0.02)
    args = ap.parse_args()

    labels = json.load(open(os.path.join(args.data, "labels.json")))
    label2id = {l: i for i, l in enumerate(labels)}
    id2label = {i: l for l, i in label2id.items()}

    df = pd.read_csv(os.path.join(args.data, "train.csv"), dtype=str, keep_default_na=False, na_filter=False)
    df["text"] = df["text"].astype(str).str.strip()
    df = df[(df["text"] != "") & df["label"].isin(label2id)].copy()
    df["labels"] = df["label"].map(label2id)
    df = df.sample(frac=1.0, random_state=13).reset_index(drop=True)
    n_val = int(len(df) * args.val_frac)
    val_df, train_df = df.iloc[:n_val], df.iloc[n_val:]
    print(f"train={len(train_df)} val={len(val_df)} labels={len(labels)}")

    tok = AutoTokenizer.from_pretrained(args.model_name)

    def encode(batch):
        return tok(batch["text"], truncation=True, max_length=args.max_length)

    train_ds = Dataset.from_pandas(train_df[["text", "labels"]], preserve_index=False).map(
        encode, batched=True, remove_columns=["text"])
    val_ds = Dataset.from_pandas(val_df[["text", "labels"]], preserve_index=False).map(
        encode, batched=True, remove_columns=["text"])

    model = AutoModelForSequenceClassification.from_pretrained(
        args.model_name, num_labels=len(labels), id2label=id2label, label2id=label2id
    )

    def compute_metrics(p):
        preds = np.argmax(p.predictions, axis=-1)
        return {"macro_f1": f1_score(p.label_ids, preds, average="macro"),
                "accuracy": float((preds == p.label_ids).mean())}

    use_cuda = torch.cuda.is_available()

    # Linear warmup over the first 6% of optimizer steps. Expressed as an
    # explicit step count because `warmup_ratio` was removed in transformers 5.x
    # while `warmup_steps` exists in every version.
    steps_per_epoch = math.ceil(len(train_ds) / args.batch_size)
    total_steps = max(1, int(round(steps_per_epoch * args.epochs)))
    warmup_steps = int(0.06 * total_steps)

    targs_kwargs = dict(
        output_dir=os.path.join(args.out, "checkpoints"),
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size * 2,
        learning_rate=args.lr,
        num_train_epochs=args.epochs,
        warmup_steps=warmup_steps,
        weight_decay=0.01,
        save_strategy="no",
        logging_steps=50,
        fp16=use_cuda,
        report_to="none",
        dataloader_num_workers=2,
    )
    # TrainingArguments fields get renamed between transformers versions
    # (evaluation_strategy -> eval_strategy, warmup_ratio removed, ...).
    # Only pass what this installed version actually accepts.
    ta_params = set(inspect.signature(TrainingArguments.__init__).parameters)
    if "eval_strategy" in ta_params:
        targs_kwargs["eval_strategy"] = "epoch"
    elif "evaluation_strategy" in ta_params:
        targs_kwargs["evaluation_strategy"] = "epoch"
    for k in [k for k in targs_kwargs if k not in ta_params]:
        print(f"note: this transformers version has no TrainingArguments.{k}; skipping it")
        targs_kwargs.pop(k)
    print(f"steps/epoch={steps_per_epoch} total_steps={total_steps} warmup_steps={warmup_steps} fp16={use_cuda}")
    targs = TrainingArguments(**targs_kwargs)

    # tokenizer kwarg was renamed processing_class in newer versions
    tr_params = set(inspect.signature(Trainer.__init__).parameters)
    tok_kwarg = {"processing_class": tok} if "processing_class" in tr_params else {"tokenizer": tok}
    trainer = Trainer(
        model=model, args=targs, train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics, **tok_kwarg,
    )
    trainer.train()
    print("validation:", trainer.evaluate())

    os.makedirs(args.out, exist_ok=True)
    trainer.save_model(args.out)
    tok.save_pretrained(args.out)
    json.dump(labels, open(os.path.join(args.out, "labels.json"), "w"), indent=2)
    assert os.path.exists(os.path.join(args.out, "config.json")), "model was not saved"
    print(f"saved to {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/evaluate.py
"""
Evaluate the fine-tuned LangID model on the clean test set and every
robustness slice. Produces per-language precision / recall / F1, macro-F1
per slice, the top confusion pairs, and a confidence-vs-accuracy table you
can use to pick the fallback-routing threshold.

Outputs:
  reports/metrics.json
  reports/REPORT.md          (paste the tables into your README)

Usage:
  python scripts/evaluate.py --data data --model model --out reports
"""
import argparse
import json
import os
from collections import Counter

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import precision_recall_fscore_support, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer

SLICES = ["clean", "short", "noisy", "code_mixed", "mixed_script"]


@torch.no_grad()
def predict(model, tok, texts, device, bs=128, max_length=64):
    probs_all = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i + bs], truncation=True, max_length=max_length,
                  padding=True, return_tensors="pt").to(device)
        logits = model(**enc).logits
        probs_all.append(torch.softmax(logits, dim=-1).cpu().numpy())
    return np.concatenate(probs_all, axis=0)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--model", default="model")
    ap.add_argument("--out", default="reports")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(args.model)
    model = AutoModelForSequenceClassification.from_pretrained(args.model).to(device).eval()
    # label order: prefer labels.json written by train.py, else the config
    # (config keys may be str or int depending on the transformers version)
    labels_path = os.path.join(args.model, "labels.json")
    if os.path.exists(labels_path):
        labels = json.load(open(labels_path))
    else:
        id2label = {int(k): v for k, v in model.config.id2label.items()}
        labels = [id2label[i] for i in range(len(id2label))]
    label2id = {l: i for i, l in enumerate(labels)}
    assert len(labels) == model.config.num_labels, "labels.json does not match the model head"
    print(f"device={device} labels={len(labels)}")

    results = {"slices": {}, "per_language_clean": {}, "top_confusions_clean": [],
               "confidence_routing_clean": []}
    md = ["# Polyglot-120 evaluation report\n"]
    md.append("| Slice | N | Macro-F1 | Accuracy |\n|---|---:|---:|---:|")

    for s in SLICES:
        path = os.path.join(args.data, f"test_{s}.csv")
        if not os.path.exists(path):
            continue
        df = pd.read_csv(path, dtype=str, keep_default_na=False, na_filter=False)
        df["text"] = df["text"].astype(str).str.strip()
        df = df[(df["text"] != "") & df["label"].isin(label2id)]
        y = df["label"].map(label2id).to_numpy()
        probs = predict(model, tok, df["text"].astype(str).tolist(), device)
        pred = probs.argmax(axis=1)
        conf = probs.max(axis=1)
        macro = f1_score(y, pred, average="macro")
        acc = float((pred == y).mean())
        results["slices"][s] = {"n": int(len(y)), "macro_f1": float(macro), "accuracy": acc}
        md.append(f"| {s} | {len(y)} | {macro:.4f} | {acc:.4f} |")

        if s == "clean":
            p, r, f, sup = precision_recall_fscore_support(y, pred, labels=list(range(len(labels))), zero_division=0)
            for i, l in enumerate(labels):
                results["per_language_clean"][l] = {"precision": float(p[i]), "recall": float(r[i]),
                                                    "f1": float(f[i]), "support": int(sup[i])}
            conf_pairs = Counter((labels[a], labels[b]) for a, b in zip(y, pred) if a != b)
            results["top_confusions_clean"] = [{"true": a, "pred": b, "count": c}
                                               for (a, b), c in conf_pairs.most_common(20)]
            for thr in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
                keep = conf >= thr
                results["confidence_routing_clean"].append({
                    "threshold": thr, "coverage": float(keep.mean()),
                    "accuracy_on_kept": float((pred[keep] == y[keep]).mean()) if keep.any() else None,
                    "accuracy_on_routed": float((pred[~keep] == y[~keep]).mean()) if (~keep).any() else None,
                })

    # worst languages on clean
    per = results["per_language_clean"]
    worst = sorted(per.items(), key=lambda kv: kv[1]["f1"])[:15]
    md.append("\n## 15 lowest-F1 languages (clean)\n")
    md.append("| Language | P | R | F1 | n |\n|---|---:|---:|---:|---:|")
    for l, m in worst:
        md.append(f"| {l} | {m['precision']:.3f} | {m['recall']:.3f} | {m['f1']:.3f} | {m['support']} |")

    md.append("\n## Top confusion pairs (clean)\n")
    md.append("| True | Predicted | Count |\n|---|---|---:|")
    for c in results["top_confusions_clean"]:
        md.append(f"| {c['true']} | {c['pred']} | {c['count']} |")

    md.append("\n## Confidence routing (clean): pick your fallback threshold\n")
    md.append("| Threshold | Coverage | Acc on kept | Acc on routed |\n|---:|---:|---:|---:|")
    for row in results["confidence_routing_clean"]:
        ak = f"{row['accuracy_on_kept']:.4f}" if row["accuracy_on_kept"] is not None else "-"
        ar = f"{row['accuracy_on_routed']:.4f}" if row["accuracy_on_routed"] is not None else "-"
        md.append(f"| {row['threshold']} | {row['coverage']:.3f} | {ak} | {ar} |")

    json.dump(results, open(os.path.join(args.out, "metrics.json"), "w"), indent=2)
    open(os.path.join(args.out, "REPORT.md"), "w", encoding="utf-8").write("\n".join(md) + "\n")
    print("\n".join(md))


if __name__ == "__main__":
    main()


## 1. Data (FLORES-200)
Downloads Meta's public tarball directly (no gating). Falls back to the Hub dataset only if that fails.

In [ ]:
!python scripts/prepare_data.py --out data --num_langs $NUM_LANGS
!head -3 data/train.csv; wc -l data/*.csv
import os; assert os.path.exists('data/train.csv') and os.path.exists('data/labels.json'), 'Data step failed. Scroll up in this cell for the error.'


## 2. Train
About 20 to 30 minutes on a T4. The cell ends with `validation: {...}` and `saved to model`.

In [ ]:
!python scripts/train.py --data data --out model --epochs $EPOCHS
import os; assert os.path.exists('model/config.json'), 'Training did not finish. Scroll up in this cell for the traceback and send it to me.'


## 3. Evaluate
These are your numbers. Copy the tables into README.md and the email.

In [ ]:
!python scripts/evaluate.py --data data --model model --out reports
import os; assert os.path.exists('reports/REPORT.md'), 'Evaluation did not finish. Scroll up in this cell for the traceback.'


## 4. Publish the model to the Hub
The Space loads it from here via `MODEL_ID`.

In [ ]:
from huggingface_hub import HfApi, create_repo
repo_id = f"{HF_USERNAME}/polyglot-120"
create_repo(repo_id, repo_type='model', exist_ok=True)
HfApi().upload_folder(folder_path='model', repo_id=repo_id, repo_type='model', ignore_patterns=['checkpoints/*'])
print('uploaded to', repo_id)


## 5. Save the report
Downloads `reports/REPORT.md` and `reports/metrics.json`. Commit them to your repo.

In [ ]:
from google.colab import files
files.download('reports/REPORT.md'); files.download('reports/metrics.json')
